# Held-out marking checklist (read-only scan)
Dataset-preparation / audit tool. **Not a model, and not evidence of extraction accuracy.** Output is machine-generated suggestions to verify visually.

Enforced in code:
- Source PDFs are opened read-only; nothing is saved back to them.
- Everything is written only under `My Drive/JEE_NEET_Extracted_Images/v4_figx/holdout_prep/` (paths are resolved first; anything outside raises `PermissionError`, and the output folder may not overlap a source folder).
- The embedded detector (`core.py` + `detect.py`) is fingerprinted and compared with the **trusted v4 baseline pinned inside this notebook**; any difference stops the run. The detector is never tuned on these files.
- All six required PDFs must exist and be unambiguous, otherwise the run stops with the reasons.
- No training, no final evaluation, no uploads.

Run Runtime > Run all (CPU is enough).

In [ ]:
import subprocess, sys
# Errors propagate: check=True raises, and pip's output is shown only on failure.
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pymupdf>=1.24', 'opencv-python-headless', 'numpy'],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise RuntimeError(f'dependency installation failed (pip exit code {r.returncode})')
import importlib
for mod in ('pymupdf', 'cv2', 'numpy'):
    try:
        m = importlib.import_module(mod)
    except Exception as e:
        raise RuntimeError(f'critical import failed after install: {mod}: {e}') from e
    print('ok', mod, getattr(m, '__version__', getattr(m, 'VersionBind', '?')))

In [ ]:
import shutil, os
shutil.rmtree('/content/holdout_pkg', ignore_errors=True)    # fresh package dir: never reuse a stale figx
os.makedirs('/content/holdout_pkg/figx')

In [ ]:
%%writefile /content/holdout_pkg/figx/__init__.py

In [ ]:
%%writefile /content/holdout_pkg/figx/core.py
"""Core geometry, rendering, annotation-box recovery and crop QA.

Coordinates: all boxes crossing module boundaries are in PDF points of the
page's visible rect (page.rect, rotation already applied), as (x0, y0, x1, y1).
Pixel boxes are only used inside a single rendered image.
"""
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from pathlib import Path

import cv2
import numpy as np
import pymupdf

Box = tuple[float, float, float, float]


# ----------------------------------------------------------------- geometry
def area(b: Box) -> float:
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])


def intersection(a: Box, b: Box) -> float:
    return area((max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])))


def iou(a: Box, b: Box) -> float:
    i = intersection(a, b)
    u = area(a) + area(b) - i
    return i / u if u > 0 else 0.0


def containment(inner: Box, outer: Box) -> float:
    """Fraction of `inner` that lies inside `outer`."""
    a = area(inner)
    return intersection(inner, outer) / a if a > 0 else 0.0


def union(a: Box, b: Box) -> Box:
    return (min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3]))


def expand(b: Box, d: float) -> Box:
    return (b[0] - d, b[1] - d, b[2] + d, b[3] + d)


def clip(b: Box, w: float, h: float) -> Box:
    return (max(0.0, b[0]), max(0.0, b[1]), min(w, b[2]), min(h, b[3]))


def suppress_duplicates(boxes: list[Box], scores: list[float] | None = None,
                        iou_thr: float = 0.85, contain_thr: float = 0.95) -> list[int]:
    """Keep-indices after removing near-identical boxes.

    Only boxes that are essentially the same region (IoU >= iou_thr, or one
    almost entirely inside the other AND similar in size) are merged, so two
    distinct neighbouring targets that merely overlap both survive.
    """
    scores = scores or [area(b) for b in boxes]
    order = sorted(range(len(boxes)), key=lambda i: -scores[i])
    keep: list[int] = []
    for i in order:
        dup = False
        for k in keep:
            a, b = boxes[i], boxes[k]
            size_ratio = min(area(a), area(b)) / max(area(a), area(b), 1e-9)
            if iou(a, b) >= iou_thr or (
                max(containment(a, b), containment(b, a)) >= contain_thr and size_ratio >= 0.6
            ):
                dup = True
                break
        if not dup:
            keep.append(i)
    return sorted(keep)


# ---------------------------------------------------------------- rendering
def render(page: pymupdf.Page, dpi: float | None = None, size: tuple[int, int] | None = None,
           clip_box: Box | None = None) -> np.ndarray:
    """Render to BGR uint8. Either dpi, or an exact (w, h) pixel size for the full page."""
    r = page.rect
    if size is not None:
        mat = pymupdf.Matrix(size[0] / r.width, size[1] / r.height)
    else:
        z = (dpi or 150) / 72.0
        mat = pymupdf.Matrix(z, z)
    pm = page.get_pixmap(matrix=mat, alpha=False, colorspace=pymupdf.csRGB,
                         clip=pymupdf.Rect(clip_box) if clip_box else None)
    img = np.frombuffer(pm.samples, np.uint8).reshape(pm.h, pm.w, 3)
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)


# ------------------------------------------------------------ red detection
@dataclass
class RedConfig:
    hue_lo: int = 10        # hue <= hue_lo  (OpenCV hue 0..179)
    hue_hi: int = 165       # or hue >= hue_hi  (covers red and maroon/crimson)
    sat_min: int = 80
    val_min: int = 50       # maroon is dark; keep low
    analysis_dpi: int = 100
    min_side_pt: float = 18.0
    min_side_support: float = 0.55   # each side must be >=55% red along its length
    max_interior_fill: float = 0.30  # solid blobs (cartoon clothing) fail this
    max_thickness_frac: float = 0.25


def red_mask(bgr: np.ndarray, cfg: RedConfig = RedConfig()) -> np.ndarray:
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    m = ((h <= cfg.hue_lo) | (h >= cfg.hue_hi)) & (s >= cfg.sat_min) & (v >= cfg.val_min)
    return m.astype(np.uint8)


@dataclass
class FrameCheck:
    ok: bool
    side_support: tuple[float, float, float, float]   # top, bottom, left, right
    interior_fill: float
    thickness_px: int
    reason: str = ""


def verify_frame(mask: np.ndarray, x: int, y: int, w: int, h: int,
                 cfg: RedConfig = RedConfig()) -> FrameCheck:
    """Is the red component inside bbox (x,y,w,h) a rectangular *frame*?

    Signals: four-side support (each border band is mostly red), low interior
    fill (frames are hollow), and a plausible stroke thickness. Rounded corners
    are tolerated because support is measured on the middle 70% of each side.
    """
    sub = mask[y:y + h, x:x + w]
    # thickness: median red run length entering from each side along middle rows/cols
    def run(line: np.ndarray) -> int:
        nz = np.flatnonzero(line == 0)
        return int(nz[0]) if nz.size else len(line)
    mids_r = range(int(h * 0.3), max(int(h * 0.7), int(h * 0.3) + 1))
    mids_c = range(int(w * 0.3), max(int(w * 0.7), int(w * 0.3) + 1))
    runs = [run(sub[r, :]) for r in mids_r] + [run(sub[r, ::-1]) for r in mids_r] + \
           [run(sub[:, c]) for c in mids_c] + [run(sub[::-1, c]) for c in mids_c]
    t = max(1, int(np.median(runs))) if runs else 1
    band = max(t + 2, 3)
    cx0, cx1 = int(w * 0.15), max(int(w * 0.85), int(w * 0.15) + 1)
    cy0, cy1 = int(h * 0.15), max(int(h * 0.85), int(h * 0.15) + 1)
    sup = (
        float(sub[:band, cx0:cx1].max(0).mean()),
        float(sub[h - band:, cx0:cx1].max(0).mean()),
        float(sub[cy0:cy1, :band].max(1).mean()),
        float(sub[cy0:cy1, w - band:].max(1).mean()),
    )
    inner = sub[band + 1:h - band - 1, band + 1:w - band - 1]
    fill = float(inner.mean()) if inner.size else 1.0
    reason = ""
    if min(sup) < cfg.min_side_support:
        reason = f"weak_side_support({min(sup):.2f})"
    elif fill > cfg.max_interior_fill:
        reason = f"solid_interior({fill:.2f})"
    elif t > cfg.max_thickness_frac * min(w, h):
        reason = f"too_thick({t}px)"
    return FrameCheck(not reason, sup, fill, t, reason)


@dataclass
class Annotation:
    page: int                 # 0-based
    box_pt: Box               # outer frame, marked-page points
    inner_pt: Box             # frame interior (border removed), marked-page points
    thickness_pt: float
    method: str               # "diff" | "single"
    check: dict = field(default_factory=dict)
    accepted: bool = True
    reason: str = ""


def _components_to_annotations(mask: np.ndarray, page_idx: int, page_rect: pymupdf.Rect,
                               cfg: RedConfig, method: str) -> list[Annotation]:
    H, W = mask.shape
    sx, sy = page_rect.width / W, page_rect.height / H
    # close small gaps in hand-drawn/compressed borders so a frame is one component
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(closed, 8)
    out: list[Annotation] = []
    min_px = cfg.min_side_pt / sx
    for k in range(1, n):
        x, y, w, h, _ = (int(v) for v in stats[k])
        if w < min_px or h < min_px:
            continue
        chk = verify_frame(closed, x, y, w, h, cfg)
        t = chk.thickness_px + 1
        box = (x * sx, y * sy, (x + w) * sx, (y + h) * sy)
        inner = ((x + t) * sx, (y + t) * sy, (x + w - t) * sx, (y + h - t) * sy)
        out.append(Annotation(page_idx, box, inner, t * sx, method, asdict(chk), chk.ok, chk.reason))
    acc = [a for a in out if a.accepted]
    keep = set(suppress_duplicates([a.box_pt for a in acc]))
    for i, a in enumerate(acc):
        if i not in keep:
            a.accepted, a.reason = False, "duplicate"
    return out


def recover_annotations(marked: pymupdf.Page, original: pymupdf.Page | None,
                        cfg: RedConfig = RedConfig()) -> tuple[list[Annotation], dict]:
    """Find human-drawn red/maroon frames on a marked page.

    With the unmarked original available ("diff" mode) only red pixels that are
    NEW in the marked page are considered, so red content that already exists in
    the figure (cartoon clothing, red graph lines) cannot become a box. Without
    an original ("single" mode) frame verification alone must reject blobs.
    Returns (annotations, diagnostics) where diagnostics includes the measured
    alignment shift between the two renders.
    """
    W = int(round(marked.rect.width * cfg.analysis_dpi / 72))
    H = int(round(marked.rect.height * cfg.analysis_dpi / 72))
    m_img = render(marked, size=(W, H))
    rm = red_mask(m_img, cfg)
    diag: dict = {"red_px_marked": int(rm.sum())}
    if original is not None:
        o_img = render(original, size=(W, H))
        g1 = cv2.cvtColor(m_img, cv2.COLOR_BGR2GRAY).astype(np.float32)
        g2 = cv2.cvtColor(o_img, cv2.COLOR_BGR2GRAY).astype(np.float32)
        (dx, dy), resp = cv2.phaseCorrelate(g2, g1)
        diag.update(shift_px=(round(dx, 2), round(dy, 2)), shift_response=round(resp, 3))
        if abs(dx) > 1 or abs(dy) > 1:   # re-register original onto marked before diffing
            o_img = cv2.warpAffine(o_img, np.float32([[1, 0, dx], [0, 1, dy]]), (W, H),
                                   borderValue=(255, 255, 255))
        ro = cv2.dilate(red_mask(o_img, cfg), np.ones((5, 5), np.uint8))
        rm = rm & (1 - ro)
        diag["red_px_added"] = int(rm.sum())
        method = "diff"
    else:
        method = "single"
    return _components_to_annotations(rm, marked.number, marked.rect, cfg, method), diag


def map_marked_to_original(box: Box, marked: pymupdf.Page, original: pymupdf.Page,
                           shift_px: tuple[float, float] = (0, 0), dpi: float = 100) -> Box:
    """Map a marked-page point box to the original page (scale + measured shift)."""
    sx = original.rect.width / marked.rect.width
    sy = original.rect.height / marked.rect.height
    dx, dy = -shift_px[0] * 72 / dpi, -shift_px[1] * 72 / dpi
    return (box[0] * sx + dx, box[1] * sy + dy, box[2] * sx + dx, box[3] * sy + dy)


# --------------------------------------------------------------- crop + QA
def tighten(img: np.ndarray, pad: int = 6, white: int = 235) -> tuple[np.ndarray, tuple[int, int, int, int]]:
    """Trim surrounding whitespace, keep `pad` px margin. Returns crop and pixel bbox."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ys, xs = np.nonzero(g < white)
    if xs.size == 0:
        return img, (0, 0, img.shape[1], img.shape[0])
    x0, x1 = max(0, xs.min() - pad), min(img.shape[1], xs.max() + 1 + pad)
    y0, y1 = max(0, ys.min() - pad), min(img.shape[0], ys.max() + 1 + pad)
    return img[y0:y1, x0:x1], (int(x0), int(y0), int(x1), int(y1))


def quality_flags(img: np.ndarray, page: pymupdf.Page, box_pt: Box, cfg: RedConfig = RedConfig()) -> dict:
    """Heuristic review flags. They guide human review; they never delete crops."""
    h, w = img.shape[:2]
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ink = float((g < 200).mean())
    edge = 3
    edge_ink = max(float((g[:edge] < 160).mean()), float((g[-edge:] < 160).mean()),
                   float((g[:, :edge] < 160).mean()), float((g[:, -edge:] < 160).mean()))
    rm = red_mask(img, cfg)
    bw = max(2, int(0.06 * min(h, w)))       # annotation remnants live at the crop border
    red_left = max(float(rm[:bw].mean()), float(rm[-bw:].mean()),
                   float(rm[:, :bw].mean()), float(rm[:, -bw:].mean()))
    # text contamination: fraction of crop area covered by text words that are part of long lines
    rect = pymupdf.Rect(box_pt)
    words = page.get_text("words", clip=rect)
    word_area = sum(area((x0, y0, x1, y1)) for x0, y0, x1, y1, *_ in words)
    text_frac = word_area / max(area(box_pt), 1e-9)
    flags = []
    if ink < 0.004:
        flags.append("empty")
    if min(h, w) < 40 or max(h, w) / max(min(h, w), 1) > 12:
        flags.append("tiny_or_narrow")
    if edge_ink > 0.35:
        flags.append("content_touches_edge")
    if red_left > 0.15:
        flags.append("red_border_remains")
    if text_frac > 0.35:
        flags.append("text_heavy")
    if ink < 0.02 and min(h, w) > 200:
        flags.append("mostly_whitespace")
    return {"ink": round(ink, 4), "edge_ink": round(edge_ink, 3), "red_frac": round(red_left, 4),
            "text_frac": round(text_frac, 3), "words": len(words), "flags": flags}


def crop_region(page: pymupdf.Page, box_pt: Box, dpi: int = 300, tight: bool = True) -> tuple[np.ndarray, Box]:
    """Render a page region; optionally trim whitespace. Returns image and final point box."""
    box_pt = clip(box_pt, page.rect.width, page.rect.height)
    img = render(page, dpi=dpi, clip_box=box_pt)
    if not tight:
        return img, box_pt
    img2, (x0, y0, x1, y1) = tighten(img)
    s = 72.0 / dpi
    return img2, (box_pt[0] + x0 * s, box_pt[1] + y0 * s, box_pt[0] + x1 * s, box_pt[1] + y1 * s)


def save_png(img: np.ndarray, path: Path, dpi: int) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(".png", img)
    if not ok:
        raise IOError(f"encode failed: {path}")
    tmp = path.with_suffix(".tmp")
    tmp.write_bytes(buf.tobytes())
    tmp.replace(path)

In [ ]:
%%writefile /content/holdout_pkg/figx/detect.py
"""Figure detection on UNMARKED pages (no human boxes).

PDF-native strategy: figures in these papers are embedded raster images and/or
vector drawings. We collect graphic elements, drop document boilerplate
(logos, barcodes, header rules repeated across pages), drop full-page scans
and table grids, cluster what remains, and attach short label text (axis
names, "10 m/s") that sits inside or right next to a cluster.

Scanned pages (one full-page image) fall back to a raster detector that
removes text-sized connected components and keeps large ink blobs; those
crops are always flagged for review.
"""
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass

import cv2
import numpy as np
import pymupdf

from .core import Box, area, containment, expand, iou, render, union


@dataclass
class DetectConfig:
    min_w_pt: float = 30.0
    min_h_pt: float = 22.0
    min_area_pt2: float = 1500.0
    max_page_frac: float = 0.60          # bigger than this = page scan / background
    merge_gap_pt: float = 9.0            # elements closer than this join one figure
    boilerplate_page_frac: float = 0.3   # element repeated on >=30% of pages = boilerplate
    margin_band_frac: float = 0.07       # header/footer band height as page fraction
    label_reach_pt: float = 14.0         # how far outside a cluster a label may sit
    label_max_chars: int = 18
    skip_tables: bool = True
    raster_fallback: bool = True
    max_aspect: float = 9.0              # wider/taller than this = banner/rule strip
    thin_pt: float = 2.5                 # stroke thickness of a "line" element
    scanned_doc_frac: float = 0.5        # raster fallback only if most pages are scans
    inline_max_h_pt: float = 36.0        # short images with text beside them are inline math


def _key(b: Box) -> tuple:
    return tuple(int(round(v / 3)) for v in b)   # 3pt tolerance


def boilerplate_keys(doc: pymupdf.Document, cfg: DetectConfig) -> set:
    """Graphic boxes that recur at the same position on many pages (logos, rules, barcodes)."""
    cnt: Counter = Counter()
    for p in doc:
        H = p.rect.height
        zone = lambda b: b[3] <= 0.15 * H or b[1] >= 0.85 * H   # header/footer zones only
        seen = set()
        for info in p.get_image_info():
            if zone(info["bbox"]):
                seen.add(_key(tuple(info["bbox"])))
        for d in p.get_drawings():
            if zone(d["rect"]):
                seen.add(_key(tuple(d["rect"])))
        cnt.update(seen)
    thr = max(3, int(cfg.boilerplate_page_frac * len(doc)))
    return {k for k, c in cnt.items() if c >= thr}


def _kind(b: Box, cfg: DetectConfig) -> str:
    w, h = b[2] - b[0], b[3] - b[1]
    if h <= cfg.thin_pt and w > cfg.thin_pt:
        return "hline"
    if w <= cfg.thin_pt and h > cfg.thin_pt:
        return "vline"
    return "shape"


def table_regions(lines: list[Box], page_w: float, page_h: float, cfg: DetectConfig) -> list[Box]:
    """Ruled tables from the network of thin horizontal/vertical strokes.

    Tables here are often drawn cell by cell, so strokes are rasterised at
    1px/pt and connected; a network with >=3 distinct horizontal rows and >=3
    distinct vertical columns is a table.
    """
    W, H = int(page_w) + 2, int(page_h) + 2
    m = np.zeros((H, W), np.uint8)
    for b in lines:
        cv2.rectangle(m, (int(b[0]), int(b[1])), (int(np.ceil(b[2])), int(np.ceil(b[3]))), 1, -1)
    n, lab, st, _ = cv2.connectedComponentsWithStats(cv2.dilate(m, np.ones((3, 3), np.uint8)), 8)
    out = []
    for k in range(1, n):
        x, y, w, h, _ = st[k]
        comp = [b for b in lines if lab[min(int(b[1]) + 1, H - 1), min(int(b[0]) + 1, W - 1)] == k]
        rows = {round(b[1] / 3) for b in comp if _kind(b, cfg) == "hline"}
        cols = {round(b[0] / 3) for b in comp if _kind(b, cfg) == "vline"}
        if len(rows) >= 3 and len(cols) >= 3 and w >= 40 and h >= 30:
            out.append((x - 1.5, y - 1.5, x + w + 1.5, y + h + 1.5))
    return out


def _graphic_elements(page: pymupdf.Page, boiler: set, cfg: DetectConfig) -> tuple[list[Box], bool]:
    W, H = page.rect.width, page.rect.height
    pa = W * H
    band = cfg.margin_band_frac * H
    els: list[Box] = []
    scanned = False
    words = page.get_text("words")
    for info in page.get_image_info():
        b = tuple(info["bbox"])
        if area(b) >= cfg.max_page_frac * pa:
            scanned = True
            continue
        if _key(b) in boiler or b[3] <= band or b[1] >= H - band:
            continue
        if _is_inline(b, words, cfg):
            continue
        els.append(b)
    for d in page.get_drawings():
        b = tuple(d["rect"])
        w, h = b[2] - b[0], b[3] - b[1]
        if _key(b) in boiler or b[3] <= band or b[1] >= H - band:
            continue
        if area(b) >= cfg.max_page_frac * pa:
            continue
        if w > 0.75 * W and h < 4:          # full-width separator rule
            continue
        if d.get("fill") is not None and w > 0.5 * W and h > 0.5 * H:   # page background
            continue
        if h <= 12 and w <= 80 and len(d["items"]) > 20:   # text drawn as vector outlines
            continue
        els.append(b)
    return els, scanned


def _is_inline(b: Box, words: list, cfg: DetectConfig) -> bool:
    """Short image sitting on a text line with words immediately left/right of it."""
    if b[3] - b[1] > cfg.inline_max_h_pt:
        return False
    for w in words:
        cy = (w[1] + w[3]) / 2
        if b[1] <= cy <= b[3] and (0 <= b[0] - w[2] <= 15 or 0 <= w[0] - b[2] <= 15):
            return True
    return False


def _cluster(boxes: list[Box], gap: float) -> list[Box]:
    clusters = [b for b in boxes]
    changed = True
    while changed:
        changed = False
        out: list[Box] = []
        for b in clusters:
            for i, c in enumerate(out):
                if area((max(b[0], c[0] - gap), max(b[1], c[1] - gap),
                         min(b[2], c[2] + gap), min(b[3], c[3] + gap))) > 0:
                    out[i] = union(b, c)
                    changed = True
                    break
            else:
                out.append(b)
        clusters = out
    return clusters


def _table_boxes(page: pymupdf.Page) -> list[Box]:
    try:
        return [tuple(t.bbox) for t in page.find_tables().tables]
    except Exception:          # find_tables is best-effort
        return []


def _attach_labels(page: pymupdf.Page, box: Box, cfg: DetectConfig) -> Box:
    """Grow the box to include short text lines (labels) inside / adjacent to it.

    Long lines are question text and are never absorbed.
    """
    reach = expand(box, cfg.label_reach_pt)
    grown = box
    d = page.get_text("dict", clip=pymupdf.Rect(reach))
    for blk in d.get("blocks", []):
        for ln in blk.get("lines", []):
            txt = "".join(s["text"] for s in ln["spans"]).strip()
            lb = tuple(ln["bbox"])
            if not txt or len(txt) > cfg.label_max_chars:
                continue
            if containment(lb, reach) < 0.9:
                continue
            grown = union(grown, lb)
    return grown


def _is_text_only(page: pymupdf.Page, box: Box) -> bool:
    words = page.get_text("words", clip=pymupdf.Rect(box))
    wa = sum(area(tuple(w[:4])) for w in words)
    return wa > 0.45 * area(box)


def detect_page_native(page: pymupdf.Page, boiler: set, cfg: DetectConfig = DetectConfig()) -> tuple[list[dict], bool]:
    els, scanned = _graphic_elements(page, boiler, cfg)
    tables: list[Box] = []
    if cfg.skip_tables:
        lines = [e for e in els if _kind(e, cfg) in ("hline", "vline")]
        tables = table_regions(lines, page.rect.width, page.rect.height, cfg) + _table_boxes(page)
        els = [e for e in els if not any(containment(e, t) > 0.9 for t in tables)]
    # cluster, remembering which element kinds each cluster holds
    clusters = _cluster(els, cfg.merge_gap_pt)
    dets = []
    for c in clusters:
        members = [e for e in els if containment(e, c) > 0.99]
        kinds = {_kind(e, cfg) for e in members}
        has_image = any(containment(tuple(i["bbox"]), c) > 0.99 for i in page.get_image_info())
        big_side = max((max(e[2] - e[0], e[3] - e[1]) for e in members if _kind(e, cfg) != "hline"), default=0)
        c = _attach_labels(page, c, cfg)
        w, h = c[2] - c[0], c[3] - c[1]
        reason = ""
        if w < cfg.min_w_pt or h < cfg.min_h_pt or area(c) < cfg.min_area_pt2:
            reason = "too_small"
        elif kinds <= {"hline"} or (not has_image and big_side < 20):
            reason = "equation_or_rule"      # fraction bars, vector accents, symbols
        elif max(w, h) / max(min(w, h), 1e-6) > cfg.max_aspect:
            reason = "banner_or_strip"
        elif area(c) > cfg.max_page_frac * page.rect.width * page.rect.height:
            reason = "too_large"
        elif any(containment(c, t) > 0.8 for t in tables):
            reason = "inside_table"
        elif any(iou(c, t) > 0.7 for t in tables):
            reason = "is_table"
        elif _is_text_only(page, c):
            reason = "text_dominated"
        dets.append({"box_pt": c, "accepted": not reason, "reason": reason, "method": "native"})
    # merge clusters that grew into each other after label attachment
    acc = [d for d in dets if d["accepted"]]
    merged = _cluster([d["box_pt"] for d in acc], 0)
    rej = [d for d in dets if not d["accepted"]]
    return [{"box_pt": b, "accepted": True, "reason": "", "method": "native"} for b in merged] + rej, scanned


def detect_page_raster(page: pymupdf.Page, cfg: DetectConfig = DetectConfig(), dpi: int = 100) -> list[dict]:
    """Fallback for scanned pages: big non-text ink blobs."""
    img = render(page, dpi=dpi)
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ink = (g < 170).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, 8)
    keep = np.zeros_like(ink)
    text_h = 0.22 * dpi   # ~16pt glyph height
    for k in range(1, n):
        x, y, w, h, a = st[k]
        if h > text_h * 1.6 or w > text_h * 6:     # bigger than a glyph/word
            keep[lab == k] = 1
    keep = cv2.dilate(keep, np.ones((15, 15), np.uint8))
    n, _, st, _ = cv2.connectedComponentsWithStats(keep, 8)
    s = 72.0 / dpi
    H, W = g.shape
    out = []
    for k in range(1, n):
        x, y, w, h, _ = st[k]
        b = (x * s, y * s, (x + w) * s, (y + h) * s)
        reason = ""
        if w * s < cfg.min_w_pt or h * s < cfg.min_h_pt:
            reason = "too_small"
        elif w > 0.9 * W and h < 0.02 * H:
            reason = "rule"
        elif w * h > cfg.max_page_frac * W * H:
            reason = "too_large"
        out.append({"box_pt": b, "accepted": not reason, "reason": reason, "method": "raster"})
    return out


def is_scanned_page(page: pymupdf.Page, cfg: DetectConfig = DetectConfig()) -> bool:
    pa = page.rect.width * page.rect.height
    big = any(area(tuple(i["bbox"])) >= cfg.max_page_frac * pa for i in page.get_image_info())
    return big and len(page.get_text("words")) < 20


def scanned_fraction(doc: pymupdf.Document, cfg: DetectConfig = DetectConfig()) -> float:
    return sum(is_scanned_page(p, cfg) for p in doc) / max(len(doc), 1)


def detect_page(page: pymupdf.Page, boiler: set, cfg: DetectConfig = DetectConfig(),
                doc_scanned_frac: float = 0.0) -> list[dict]:
    """doc_scanned_frac: share of scanned pages in this document. Isolated scanned
    pages in a digital document are covers/instructions and are skipped."""
    dets, _ = detect_page_native(page, boiler, cfg)
    if (cfg.raster_fallback and doc_scanned_frac >= cfg.scanned_doc_frac
            and is_scanned_page(page, cfg) and not any(d["accepted"] for d in dets)):
        return detect_page_raster(page, cfg)
    return dets

In [ ]:
%%writefile /content/holdout_pkg/figx/scan.py
"""Read-only page scan used to prepare the held-out marking checklist.

This is a dataset-preparation/audit tool. It is NOT a model and its output is
NOT evidence of extraction accuracy.

Guarantees
- Source PDFs are only opened for reading; nothing here saves or edits them.
- Every output goes through `safe_write_text`, which rejects paths that resolve
  outside the configured output directory (including `..` and symlink escapes).
- The detector is fingerprinted with `detector_fingerprint` and compared with a
  pinned trusted baseline (see FINGERPRINT SPEC below), not with itself.
- Page selections are machine-generated suggestions that need visual confirmation.

FINGERPRINT SPEC (spec id "figx-fp-v2")
- Files: figx/core.py and figx/detect.py (the production detector). scan.py,
  pipeline.py and __init__.py are NOT part of it.
- Source normalisation, applied identically everywhere: decode UTF-8, drop a
  leading BOM, convert CRLF and lone CR to LF, strip trailing spaces/tabs on
  EVERY line, drop trailing empty lines, end with exactly one LF. Indentation,
  comments, blank lines inside the file and all other characters are kept, so
  any meaningful source change (including comment edits) changes the hash.
- Config: default DetectConfig and RedConfig values, serialised as sorted JSON.
  This is taken from the *imported* classes, so it also catches a module that
  was patched at runtime.
- Hash: SHA-256 per file, of the config JSON, and a combined SHA-256 over
  "spec|core.py:<h>|detect.py:<h>|config:<h>".
"""
from __future__ import annotations

import dataclasses
import hashlib
import json
from pathlib import Path

import cv2
import numpy as np
import pymupdf

from .core import RedConfig, render
from .detect import DetectConfig, boilerplate_keys, detect_page, is_scanned_page, scanned_fraction

FP_SPEC = "figx-fp-v2"
FP_FILES = ("core.py", "detect.py")
WARM_MIN = 0.003           # share of red/orange pixels that makes a page a colour-risk page
MIN_SIG_CHARS = 40         # fewer alphanumeric chars than this = no usable text signature
THUMB_SIDE = 16            # 16x16 average hash = 256 bits
THUMB_HAMMING_MAX = 20     # of 256 bits: below this two sampled pages "look the same"
THUMB_MIN_STD = 4.0        # near-uniform pages (blank/white scans) give meaningless hashes


class MissingFilesError(RuntimeError):
    pass


class FingerprintMismatch(RuntimeError):
    pass


# ------------------------------------------------------------ provenance
def normalize_source(text: str) -> str:
    text = text.lstrip("﻿").replace("\r\n", "\n").replace("\r", "\n")
    lines = [ln.rstrip(" \t") for ln in text.split("\n")]
    while lines and lines[-1] == "":
        lines.pop()
    return "\n".join(lines) + "\n"


def _sha256(s: str) -> str:
    return hashlib.sha256(s.encode("utf8")).hexdigest()


def config_json() -> str:
    return json.dumps({"DetectConfig": dataclasses.asdict(DetectConfig()),
                       "RedConfig": dataclasses.asdict(RedConfig())}, sort_keys=True, separators=(",", ":"))


def fingerprint_from_sources(sources: dict[str, str], cfg_json: str) -> dict:
    files = {n: _sha256(normalize_source(sources[n])) for n in FP_FILES}
    cfg = _sha256(cfg_json)
    combined = _sha256(f"{FP_SPEC}|" + "|".join(f"{n}:{files[n]}" for n in FP_FILES) + f"|config:{cfg}")
    return {"spec": FP_SPEC, "files": files, "config": cfg, "combined": combined}


def detector_fingerprint(pkg_dir: Path) -> dict:
    """Fingerprint of the detector source in `pkg_dir` plus the config of the imported classes."""
    pkg_dir = Path(pkg_dir)
    return fingerprint_from_sources({n: (pkg_dir / n).read_text(encoding="utf8") for n in FP_FILES}, config_json())


def compare_fingerprints(actual: dict, expected: dict) -> list[str]:
    """Human-readable list of mismatches (empty list = identical)."""
    out = []
    if actual.get("spec") != expected.get("spec"):
        out.append(f"spec {actual.get('spec')} != {expected.get('spec')}")
    for n in FP_FILES:
        if actual["files"].get(n) != expected["files"].get(n):
            out.append(f"{n} source differs")
    if actual.get("config") != expected.get("config"):
        out.append("default DetectConfig/RedConfig values differ")
    return out


def require_fingerprint(actual: dict, expected: dict, label: str) -> None:
    diff = compare_fingerprints(actual, expected)
    if diff:
        raise FingerprintMismatch(f"{label}: detector does not match the trusted v4 baseline: {'; '.join(diff)}")


def notebook_sources(nb_path: Path, prefix: str) -> dict[str, str]:
    """Extract `%%writefile <prefix><name>` cell bodies from a notebook."""
    nb = json.loads(Path(nb_path).read_text(encoding="utf8"))
    out = {}
    for c in nb["cells"]:
        src = "".join(c.get("source", []))
        head, _, body = src.partition("\n")
        if head.startswith("%%writefile " + prefix):
            out[head.split(prefix, 1)[1].strip()] = body
    missing = [n for n in FP_FILES if n not in out]
    if missing:
        raise FingerprintMismatch(f"{nb_path} has no embedded {missing}")
    return out


# ------------------------------------------------------------ file access
def sha1_file(path: Path) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while b := f.read(1 << 20):
            h.update(b)
    return h.hexdigest()


def locate(targets: list[str], roots: list[Path]) -> dict:
    """Find target filenames under the roots. Never raises; the caller decides."""
    index: dict[str, list[Path]] = {}
    root_report = []
    for r in roots:
        r = Path(r)
        if r.exists():
            pdfs = sorted(r.rglob("*.pdf"))
            root_report.append({"root": str(r), "accessible": True, "pdfs": len(pdfs)})
            for p in pdfs:
                index.setdefault(p.name, []).append(p)
        else:
            root_report.append({"root": str(r), "accessible": False, "pdfs": 0})
    found, ambiguous = {}, {}
    for t in targets:
        if t not in index:
            continue
        paths = index[t]
        if len({sha1_file(p) for p in paths}) > 1:     # same name, different content: refuse to guess
            ambiguous[t] = [str(p) for p in paths]
        else:
            found[t] = paths[0]
    missing = [t for t in targets if t not in index]
    return {"found": found, "missing": missing, "ambiguous": ambiguous, "roots": root_report}


def require_all(loc: dict, required: list[str]) -> None:
    missing = [t for t in required if t in loc["missing"]]
    ambiguous = [t for t in required if t in loc["ambiguous"]]
    if missing or ambiguous:
        roots = "; ".join(f"{r['root']} ({'accessible, %d PDFs' % r['pdfs'] if r['accessible'] else 'NOT accessible'})"
                          for r in loc["roots"])
        parts = []
        if missing:
            parts.append(f"Missing: {', '.join(missing)}")
        if ambiguous:
            parts.append("Ambiguous (same name, different content): " +
                         "; ".join(f"{t} -> {loc['ambiguous'][t]}" for t in ambiguous))
        raise MissingFilesError(
            f"{len(missing) + len(ambiguous)} required PDF(s) unusable, no checklist produced.\n"
            + "\n".join(parts) + f"\nDrive roots searched: {roots}\n"
            "Add shortcuts to the shared JEE/NEET folders under My Drive, fix the filenames, "
            "or remove the conflicting copy.")


def assert_inside(path: Path, out_dir: Path) -> Path:
    p, o = Path(path).resolve(), Path(out_dir).resolve()
    if o != p and o not in p.parents:
        raise PermissionError(f"refusing to write outside {o}: {p}")
    return p


def check_output_dir(out_dir: Path, source_roots: list[Path]) -> Path:
    """The output dir must not be inside a source root, nor contain one."""
    o = Path(out_dir).resolve()
    for r in source_roots:
        r = Path(r).resolve()
        if o == r or r in o.parents or o in r.parents:
            raise PermissionError(f"output dir {o} overlaps source root {r}")
    return o


def plan_drive_mount(candidates: list[Path]) -> tuple[str, Path]:
    """Decide how to expose Drive without mounting over a non-empty directory.

    Returns ("use", p) if Drive is already mounted at p (contains MyDrive), or
    ("mount", p) for the first candidate that is missing or empty. Raises when
    every candidate is a non-empty directory without MyDrive.
    """
    for c in map(Path, candidates):
        if (c / "MyDrive").exists():
            return "use", c
    for c in map(Path, candidates):
        if not c.exists() or not any(c.iterdir()):
            return "mount", c
    raise RuntimeError("no safe Drive mount point: " + ", ".join(f"{c} is non-empty" for c in candidates))


def safe_write_text(path: Path, text: str, out_dir: Path) -> Path:
    p = assert_inside(path, out_dir)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + ".tmp")
    tmp.write_text(text, encoding="utf8")
    tmp.replace(p)
    return p


# ------------------------------------------------------------ page signals
def warm_share(page, dpi=40) -> float:
    """Share of pixels that are red/orange (hue<=25 or >=165, saturated): false-positive risk."""
    img = render(page, dpi=dpi)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    return float((((h <= 25) | (h >= 165)) & (s >= 90) & (v >= 60)).mean())


def text_signature(doc, n_pages=4) -> str | None:
    """SHA-1 of the first pages' words, or None when there is no meaningful text
    (scanned PDFs). None never counts as a match."""
    words = []
    for p in list(doc)[:n_pages]:
        words += [w[4] for w in p.get_text("words")]
    text = " ".join(words)
    if sum(ch.isalnum() for ch in text) < MIN_SIG_CHARS:
        return None
    return hashlib.sha1(text.encode("utf8", "ignore")).hexdigest()[:12]


def page_ahash(page) -> str | None:
    """16x16 average hash of a low-res render; None for near-uniform pages."""
    g = cv2.cvtColor(render(page, dpi=30), cv2.COLOR_BGR2GRAY)
    if float(g.std()) < THUMB_MIN_STD:
        return None
    g = cv2.resize(g, (THUMB_SIDE, THUMB_SIDE), interpolation=cv2.INTER_AREA)
    bits = "".join("1" if b else "0" for b in (g > g.mean()).flatten())
    return format(int(bits, 2), f"0{THUMB_SIDE * THUMB_SIDE // 4}x")


def sample_pages(n: int) -> list[int]:
    return sorted({0, n // 4, n // 2, (3 * n) // 4, n - 1}) if n else []


def hamming(a: str, b: str) -> int:
    return bin(int(a, 16) ^ int(b, 16)).count("1")


def scan_pdf(path: Path, cfg: DetectConfig = DetectConfig()) -> dict:
    """Never raises. status: 'ok' | 'partial' (some pages failed) | 'error' (could not open/scan)."""
    path = Path(path)
    out = {"file": path.name, "path": str(path), "status": "ok", "error": None, "n_pages": 0, "sha1": None,
           "bytes": None, "scanned_frac": 0.0, "text_sig": None, "thumbs": [], "pages": [],
           "pages_ok": 0, "pages_failed": 0}
    try:
        out["sha1"] = sha1_file(path)
        out["bytes"] = path.stat().st_size
        with pymupdf.open(path) as d:           # read-only; never saved
            if d.needs_pass:
                raise ValueError("encrypted")
            out["n_pages"] = len(d)
            boiler = boilerplate_keys(d, cfg)
            sf = scanned_fraction(d, cfg)
            out["scanned_frac"] = round(sf, 2)
            out["text_sig"] = text_signature(d)
            out["thumbs"] = [page_ahash(d[i]) for i in sample_pages(len(d))]
            for i, p in enumerate(d):
                rec = {"page": i + 1, "status": "ok", "error": None}
                try:
                    dets = [x for x in detect_page(p, boiler, cfg, sf) if x["accepted"]]
                    drawings = len(p.get_drawings())
                    has_img = bool(p.get_image_info())
                    kind = ("scanned_page" if is_scanned_page(p, cfg) else
                            "none" if not dets else
                            "raster+vector" if has_img and drawings > 50 else
                            "raster" if has_img else "vector")
                    rec.update(n_figs=len(dets), kind=kind, warm_share=round(warm_share(p), 4),
                               words=len(p.get_text("words")), methods=sorted({x["method"] for x in dets}),
                               boxes=[[round(v) for v in x["box_pt"]] for x in dets])
                except Exception as e:           # one bad page must not hide the rest
                    rec.update(status="error", error=f"{type(e).__name__}: {e}", n_figs=0, kind="unknown",
                               warm_share=0.0, words=0, methods=[], boxes=[])
                out["pages"].append(rec)
        out["pages_failed"] = sum(p["status"] == "error" for p in out["pages"])
        out["pages_ok"] = len(out["pages"]) - out["pages_failed"]
        if out["pages_failed"]:
            out["status"] = "partial"
            out["error"] = f"{out['pages_failed']} of {out['n_pages']} page(s) failed"
    except Exception as e:
        out["status"], out["error"] = "error", f"{type(e).__name__}: {e}"
    return out


# ------------------------------------------------------------ selection
def _spread(items: list, k: int) -> list:
    if k <= 0 or not items:
        return []
    if len(items) <= k:
        return list(items)
    idx = sorted(set(np.linspace(0, len(items) - 1, k).round().astype(int)))
    return [items[i] for i in idx]


def select_pages(scan: dict, n_fig=8, n_neg=2, n_warm=1, n_scanned=3) -> dict:
    """Machine-generated suggestions. Returns {'selections': [...], 'shortages': [...]}.

    A detector result of 0 never proves a page has no figure; negatives are
    only 'plausible' and the user must confirm visually.
    """
    ok = [p for p in scan["pages"] if p["status"] == "ok"]
    sel: dict[int, dict] = {}
    short: list[str] = []

    def add(p, role, reason):
        e = sel.setdefault(p["page"], {"page": p["page"], "roles": [], "reasons": [], "rec": p})
        e["roles"].append(role)
        e["reasons"].append(reason)

    figs = [p for p in ok if p["n_figs"] > 0]
    for p in _spread(figs, n_fig):
        add(p, "figure_candidate", f"detector found {p['n_figs']} candidate figure(s) ({p['kind']}); spread across the document")
    if len(figs) < n_fig:
        short.append(f"only {len(figs)} page(s) have detector figure candidates (wanted ~{n_fig}); "
                     "the detector may be missing figures on this document, so inspect it by eye")

    scanned = [p for p in ok if p["kind"] == "scanned_page"]
    if scanned:
        for p in _spread([q for q in scanned if q["page"] not in sel], n_scanned):
            add(p, "scanned_sample", "scanned page (no text layer): detector uses the raster fallback, which is "
                                     "unvalidated; mark figures by eye")
    elif scan.get("scanned_frac", 0) > 0:
        short.append("scanned share > 0 but no individual scanned page identified")

    digital_neg = [p for p in ok if p["n_figs"] == 0 and p["words"] > 40 and p["kind"] != "scanned_page"
                   and p["page"] not in sel]
    pool, why = digital_neg, "text page, detector found 0 candidates: PLAUSIBLE negative, not confirmed"
    if len(pool) < n_neg:
        extra = [p for p in ok if p["n_figs"] == 0 and p["page"] not in sel and p not in pool]
        pool = pool + extra
        if extra:
            why = "detector found 0 candidates (low-text/scanned pages included): PLAUSIBLE negative, not confirmed"
    for p in _spread(pool, n_neg):
        add(p, "negative_control", why)
    if len(pool) < n_neg:
        short.append(f"only {len(pool)} plausible negative-control page(s) available (wanted {n_neg})")

    warm = sorted([p for p in ok if p["warm_share"] >= WARM_MIN], key=lambda p: -p["warm_share"])[:n_warm]
    for p in warm:
        add(p, "colour_control", f"red/orange content on {p['warm_share'] * 100:.1f}% of the page "
                                 f"({'detector also found a figure' if p['n_figs'] else 'no detector figure'})")
    if len(warm) < n_warm:
        short.append(f"no page reaches the red/orange threshold ({WARM_MIN * 100:.1f}% of pixels); "
                     "no colour-control page can be suggested")

    bad = [p["page"] for p in scan["pages"] if p["status"] == "error"]
    if bad:
        short.append(f"{len(bad)} page(s) failed to scan and could not be considered: {bad[:20]}")
    return {"selections": [sel[k] for k in sorted(sel)], "shortages": short}


# ------------------------------------------------------------ duplicates
def classify_pair(a: dict, b: dict) -> dict:
    """Evidence-based duplicate classification. Only identical bytes are 'confirmed'.

    Missing evidence (no text signature, uniform pages) is never treated as a match.
    """
    if a["sha1"] and a["sha1"] == b["sha1"]:
        return {"a": a["file"], "b": b["file"], "level": "confirmed_identical_bytes", "evidence": ["identical SHA-1"]}
    ev = []
    sig = a["text_sig"] is not None and a["text_sig"] == b["text_sig"]
    same_pages = a["n_pages"] == b["n_pages"] and a["n_pages"] > 0
    pairs = [(x, y) for x, y in zip(a["thumbs"], b["thumbs"]) if x is not None and y is not None]
    comparable = len(a["thumbs"]) == len(b["thumbs"]) and len(pairs) >= max(1, len(a["thumbs"]) // 2)
    visual = comparable and all(hamming(x, y) <= THUMB_HAMMING_MAX for x, y in pairs)
    if sig:
        ev.append("same first-pages text signature")
    if a["text_sig"] is None or b["text_sig"] is None:
        ev.append("text signature unavailable (no extractable text)")
    if same_pages:
        ev.append(f"same page count ({a['n_pages']})")
    if visual:
        ev.append(f"{len(pairs)} sampled page(s) look visually similar")
    elif not comparable:
        ev.append("too few non-blank sampled pages to compare visually")
    both_text = a["text_sig"] is not None and b["text_sig"] is not None
    if both_text and not sig:
        ev.append("text signatures differ (different content)")      # visual hashes of near-white pages are unreliable
        level = "distinct"
    elif sig and same_pages and visual:
        level = "suspected_strong"
    elif sig:
        level = "suspected_text_only"
    elif same_pages and visual and not both_text:
        level = "suspected_visual_only"                              # only when text evidence is unavailable
    else:
        level = "distinct"
    return {"a": a["file"], "b": b["file"], "level": level, "evidence": ev}


def duplicate_report(scans: list[dict]) -> list[dict]:
    ok = [s for s in scans if s["status"] in ("ok", "partial")]
    out = []
    for i, a in enumerate(ok):
        for b in ok[i + 1:]:
            c = classify_pair(a, b)
            if c["level"] != "distinct":
                out.append(c)
    return out


def duplicate_wording(d: dict) -> str:
    if d["level"] == "confirmed_identical_bytes":
        return f"CONFIRMED byte-identical: {d['a']} == {d['b']}"
    tag = {"suspected_strong": "STRONG suspected duplicate", "suspected_text_only": "WEAK suspected duplicate (text only)",
           "suspected_visual_only": "WEAK suspected duplicate (visual only)"}[d["level"]]
    return f"{tag} (NOT proof): {d['a']} ~ {d['b']} [{'; '.join(d['evidence'])}]"


# ------------------------------------------------------------ summary + checklist
def build_summary(targets: list[str], loc: dict, scans: list[dict], dups: list[dict],
                  fingerprint_ok: bool | None, extra_warnings=()) -> dict:
    by = {s["file"]: s for s in scans}
    located = [t for t in targets if t in loc["found"]]
    opened = [t for t in located if t in by and by[t]["status"] in ("ok", "partial")]
    partial = [t for t in opened if by[t]["status"] == "partial"]
    failed_open = {t: by[t]["error"] for t in located if t in by and by[t]["status"] == "error"}
    omitted = [t for t in targets if t not in opened]
    s = {
        "expected": len(targets),
        "pdfs_located": len(located),
        "pdfs_opened_and_scanned": len(opened),
        "pdfs_fully_scanned": len(opened) - len(partial),
        "pdfs_with_page_errors": partial,
        "pdfs_failed_to_open": failed_open,
        "pdfs_not_scanned": [t for t in located if t not in by],
        "pdfs_missing": [t for t in targets if t in loc["missing"]],
        "pdfs_ambiguous": [t for t in targets if t in loc.get("ambiguous", {})],
        "omitted": omitted,
        "pages_processed_ok": sum(by[t]["pages_ok"] for t in opened),
        "pages_failed": sum(by[t]["pages_failed"] for t in opened),
        "confirmed_duplicates": [duplicate_wording(d) for d in dups if d["level"] == "confirmed_identical_bytes"],
        "suspected_duplicates": [duplicate_wording(d) for d in dups if d["level"] != "confirmed_identical_bytes"],
        "fingerprint_verified": fingerprint_ok,
        "warnings": list(extra_warnings),
    }
    s["complete"] = not omitted and not partial and s["pages_failed"] == 0 and fingerprint_ok is True
    return s


def summary_lines(s: dict) -> list[str]:
    fp = {True: "VERIFIED against trusted v4 baseline", False: "MISMATCH", None: "NOT CHECKED"}[s["fingerprint_verified"]]
    L = [f"- Status: **{'COMPLETE' if s['complete'] else 'INCOMPLETE: do not rely on this checklist as-is'}**",
         f"- Detector fingerprint: **{fp}**",
         f"- PDFs expected {s['expected']} | located {s['pdfs_located']} | opened & scanned {s['pdfs_opened_and_scanned']}"
         f" | fully scanned {s['pdfs_fully_scanned']} | with page errors {len(s['pdfs_with_page_errors'])}"
         f" | omitted {len(s['omitted'])}",
         f"- Pages processed OK {s['pages_processed_ok']} | pages failed {s['pages_failed']}"]
    if s["pdfs_with_page_errors"]:
        L.append(f"- PDFs with page-level errors: {', '.join(s['pdfs_with_page_errors'])}")
    for f, e in s["pdfs_failed_to_open"].items():
        L.append(f"- Failed to open/scan: {f}: {e}")
    if s["pdfs_missing"]:
        L.append(f"- Missing: {', '.join(s['pdfs_missing'])}")
    if s["pdfs_ambiguous"]:
        L.append(f"- Ambiguous (same name, different content): {', '.join(s['pdfs_ambiguous'])}")
    if s["omitted"]:
        L.append(f"- Omitted: {', '.join(s['omitted'])}")
    L += [f"- {d}" for d in s["confirmed_duplicates"] + s["suspected_duplicates"]] or \
         ["- No duplicate evidence among the scanned PDFs."]
    L += [f"- Warning: {w}" for w in s["warnings"]]
    return L


def render_checklist(summary: dict, scans: list[dict], selections: dict, fingerprint: dict, replacements=()) -> str:
    L = ["# Marking checklist (held-out set)", "",
         "> **MACHINE-GENERATED SUGGESTIONS. Verify every page visually.** Detector results are hints: a page "
         "with 0 detections is *not* proven figure-free, and a detected figure may be wrong. If you see a "
         "figure the detector missed, mark it anyway. Those misses are what the evaluation should reveal. "
         "This scan is a preparation tool, not evidence of extraction accuracy.", "",
         f"Detector fingerprint `{fingerprint['combined'][:16]}` (spec {fingerprint['spec']}).", "",
         "## Run summary", ""] + summary_lines(summary)
    for r in replacements:
        L.append(f"- Replacement SUGGESTION (needs your approval; never applied automatically): {r}")
    L += ["", "How to mark: draw one red/maroon rectangle around EACH target figure on the pages marked "
          "**MARK**. Save as `<name> red boxes marked.pdf` beside an untouched copy of the original in "
          "`Reference_PDFs/`. Do not crop, rotate or reorder pages.", ""]
    for s in scans:
        L += [f"## {s['file']}"]
        if s["status"] == "error":
            L += [f"**Not scanned:** {s['error']}", ""]
            continue
        sel = selections[s["file"]]
        L += [f"{s['n_pages']} pages | scanned share {s['scanned_frac']} | SHA-1 `{s['sha1'][:12]}` | "
              f"text signature {'available' if s['text_sig'] else 'UNAVAILABLE (no extractable text)'}", ""]
        if s["status"] == "partial":
            L += [f"⚠ **Partially scanned:** {s['error']}. Suggestions below ignore the failed pages.", ""]
        if s["scanned_frac"] > 0:
            L += ["⚠ Contains scanned pages. The raster fallback ran but has **not been validated** on real data.", ""]
        for sh in sel["shortages"]:
            L.append(f"- ⚠ Shortage: {sh}")
        if sel["shortages"]:
            L.append("")
        L += ["| Page | Action | Role | Candidates | Visual type | Scan status | Red/orange risk | Why (suggestion) |",
              "|---|---|---|---|---|---|---|---|"]
        for e in sel["selections"]:
            r = e["rec"]
            roles = "+".join(e["roles"])
            if "negative_control" in e["roles"]:
                act = "Leave unmarked ONLY if you confirm no target figure"
            elif "colour_control" in e["roles"]:
                act = "Do NOT box red/orange content that is not a target figure; mark a figure only if it truly is one"
            else:
                act = "MARK every target figure"
            risk = f"{r['warm_share'] * 100:.1f}%" + (" ⚠" if r["warm_share"] >= WARM_MIN else "")
            L.append(f"| {e['page']} | {act} | {roles} | {r['n_figs']} | {r['kind']} | {r['status']} | {risk} | "
                     f"{' / '.join(e['reasons'])} |")
        L.append("")
    return "\n".join(L)


def holdout_manifest(targets: list[str], scans: list[dict], fingerprint: dict) -> dict:
    """Frozen identity of the held-out set (by content hash) for leakage checks in later phases."""
    by = {s["file"]: s for s in scans}
    return {"split": "heldout_test", "purpose": "final evaluation only; never train/tune/select on these",
            "detector_fingerprint": fingerprint["combined"],
            "documents": [{"file": t, "sha1": by[t]["sha1"] if t in by else None,
                           "bytes": by[t]["bytes"] if t in by else None,
                           "n_pages": by[t]["n_pages"] if t in by else None} for t in targets]}

In [ ]:
import sys
for m in [m for m in sys.modules if m == 'figx' or m.startswith('figx.')]:
    del sys.modules[m]                                   # never reuse a previously imported figx
sys.path.insert(0, '/content/holdout_pkg')
from pathlib import Path
import figx, figx.scan as scan
assert Path(figx.__file__).resolve().parent == Path('/content/holdout_pkg/figx'), figx.__file__

# Trusted v4 baseline, pinned at build time from jee_neet_figx_v4.ipynb (NOT computed from these files).
TRUSTED_V4 = {"spec": "figx-fp-v2", "files": {"core.py": "7363aff7cf8331aba27ee5ad584a6a38599a7108d0f2c8621d623db77fee2197", "detect.py": "14af63a3fc1bffe5ed47f327cfbc080b3fbc12c003b1563b1cb788cd3328b42c"}, "config": "8ca9444a073717683b8b770a280c69574bf31a84e9f411a96e20acf74e74ed4f", "combined": "069368f0ad53c256440369774ccffc3b391a6a64c0120675278bf0ef9845c4bf"}
FP = scan.detector_fingerprint(Path('/content/holdout_pkg/figx'))
scan.require_fingerprint(FP, TRUSTED_V4, 'embedded detector')      # raises FingerprintMismatch on any difference
FINGERPRINT_OK = True
print('Detector fingerprint VERIFIED against trusted v4 baseline:', FP['combined'][:16])

In [ ]:
action, M = scan.plan_drive_mount([Path('/content/drive'), Path('/content/gdrive')])
print('Drive:', action, M)
if action == 'mount':
    from google.colab import drive
    drive.mount(str(M))                                  # only ever mounts at an absent/empty directory
DRIVE = M / 'MyDrive'
assert DRIVE.exists(), f'{DRIVE} not found after mount'

In [ ]:
ROOTS = [DRIVE / 'JEE', DRIVE / 'NEET', DRIVE / 'Reference_PDFs']
TARGETS = ['Copy of original (24).pdf', 'Copy of original (33).pdf', 'Copy of original (41).pdf',
           'Copy of NEET 2016.pdf', 'Copy of NEET 2024.pdf', 'Copy of NEET 2005.pdf']
JEE_FALLBACKS = ['Copy of original (14).pdf', 'Copy of original (46).pdf', 'Copy of original (11).pdf',
                 'Copy of original (12).pdf', 'Copy of original (5).pdf', 'Copy of original (6).pdf']
OUT = scan.check_output_dir(DRIVE / 'JEE_NEET_Extracted_Images' / 'v4_figx' / 'holdout_prep', ROOTS)

loc = scan.locate(TARGETS + JEE_FALLBACKS, ROOTS)
for r in loc['roots']:
    print(('OK      ' if r['accessible'] else 'MISSING ') + r['root'], '-', r['pdfs'], 'PDFs')
scan.require_all(loc, TARGETS)       # raises MissingFilesError listing missing / ambiguous files
print('All', len(TARGETS), 'required PDFs located and unambiguous. Output ->', OUT)

In [ ]:
import time
scans = []
for t in TARGETS:
    t0 = time.time()
    s = scan.scan_pdf(loc['found'][t])
    scans.append(s)
    print(f"{t}: {s['status']} | {s['n_pages']} pages ({s['pages_ok']} ok, {s['pages_failed']} failed) | scanned share {s['scanned_frac']} | "
          f"{sum(p.get('n_figs', 0) > 0 for p in s['pages'])} pages with candidates | {time.time() - t0:.1f}s", s['error'] or '')

In [ ]:
dups = scan.duplicate_report(scans)
for d in dups:
    print(scan.duplicate_wording(d))
if not dups:
    print('No duplicate evidence among the six.')

# Replacement SUGGESTIONS only (never applied). Triggered by confirmed OR suspected duplicates.
repl_scans, replacements, extra_warnings = [], [], []
for d in dups:
    victim = max(d['a'], d['b'], key=TARGETS.index)
    chosen = None
    for fb in [f for f in JEE_FALLBACKS if f in loc['found'] and f not in [s['file'] for s in repl_scans]]:
        cand = scan.scan_pdf(loc['found'][fb])
        others = [s for s in scans if s['file'] != victim] + repl_scans
        clash = [x for x in scan.duplicate_report(others + [cand]) if fb in (x['a'], x['b'])]
        if cand['status'] in ('ok', 'partial') and not clash:
            chosen = cand
            break
    if chosen:
        repl_scans.append(chosen)
        replacements.append(f"replace {victim} with {chosen['file']} (no duplicate evidence against the other scanned files). "
                            f"Reason: {scan.duplicate_wording(d)}")
    else:
        extra_warnings.append(f"duplicate evidence for {victim} but no usable distinct JEE fallback found among {JEE_FALLBACKS}")
print(replacements or 'No replacement suggested.', extra_warnings)

In [ ]:
import json
all_scans = scans + repl_scans
selections = {s['file']: scan.select_pages(s) if s['status'] != 'error' else {'selections': [], 'shortages': [s['error']]}
              for s in all_scans}
summary = scan.build_summary(TARGETS, loc, scans, dups, FINGERPRINT_OK, extra_warnings)
md_text = scan.render_checklist(summary, all_scans, selections, FP, replacements)
scan.safe_write_text(OUT / 'marking_checklist.md', md_text, OUT)
payload = {'detector_fingerprint': FP, 'trusted_baseline': TRUSTED_V4['combined'], 'summary': summary, 'duplicates': dups,
           'replacement_suggestions': replacements,
           'selections': {f: {'shortages': v['shortages'],
                              'pages': [{'page': e['page'], 'roles': e['roles'], 'reasons': e['reasons']} for e in v['selections']]}
                          for f, v in selections.items()},
           'scans': all_scans}
scan.safe_write_text(OUT / 'page_scan.json', json.dumps(payload, indent=1), OUT)
scan.safe_write_text(OUT / 'holdout_manifest.json', json.dumps(scan.holdout_manifest(TARGETS, scans, FP), indent=1), OUT)

print('=' * 64)
print('COMPLETION SUMMARY')
print('\n'.join(scan.summary_lines(summary)))
for f, v in selections.items():
    for sh in v['shortages']: print(f'  shortage [{f}]:', sh)
print('Written to:', OUT)
print('=' * 64)
from IPython.display import Markdown, display
display(Markdown(md_text))

Suggestions only: a page with 0 detections is not proven figure-free, and missed or wrong detections are what the later evaluation should reveal. Mark copies, never the source PDFs. Replacement suggestions need your explicit approval.